# 02 — PAO Baseline Classification (TF-IDF + Logistic Regression)

Trains and evaluates leakage-free linear baselines on `data/processed/pao_metadata_v2.parquet` using 5-fold `StratifiedGroupKFold` (`groups = group_id`, `seed = 42`), matching `configs/classification/tfidf_logreg.yaml` and `configs/eval/classification.yaml`.


In [1]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

pd.set_option("display.max_colwidth", 140)

DATA_FILE = Path("../data/processed/pao_metadata_v2.parquet")
RUN_DIR = Path("../models/classification/tfidf_logreg_v2")

df = pd.read_parquet(DATA_FILE)
# Exclude unresolved rows waiting on PAO domain confirmation
df = df[df["review_status"] != "needs_domain_review"].reset_index(drop=True)

print("Rows for modeling:", len(df))
print("\nClass distribution:")
print(df["label"].value_counts())
print("\nIdentical-text conflict rows (CLEARANCE vs LEAVES):", int(df["has_label_conflict"].sum()))

Rows for modeling: 3618

Class distribution:
label
COMMUNICATION    2111
CLEARANCE         693
LEAVES            652
AOsMEMOs          162
Name: count, dtype: int64

Identical-text conflict rows (CLEARANCE vs LEAVES): 354


In [2]:
# Shared 5-fold StratifiedGroupKFold splitter + explicit zero-leakage assertion
X = df["document_text"]
y = df["label"]
groups = df["group_id"]

cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (tr_idx, va_idx) in enumerate(cv.split(X, y, groups), start=1):
    overlap = set(groups.iloc[tr_idx]).intersection(set(groups.iloc[va_idx]))
    assert len(overlap) == 0, f"Leakage detected in fold {fold}!"
    val_counts = y.iloc[va_idx].value_counts().to_dict()
    print(f"Fold {fold}: train={len(tr_idx):4d}, val={len(va_idx):3d}, val_classes={val_counts}")

Fold 1: train=2844, val=774, val_classes={'COMMUNICATION': 422, 'CLEARANCE': 208, 'LEAVES': 112, 'AOsMEMOs': 32}
Fold 2: train=2914, val=704, val_classes={'COMMUNICATION': 422, 'LEAVES': 129, 'CLEARANCE': 121, 'AOsMEMOs': 32}
Fold 3: train=2857, val=761, val_classes={'COMMUNICATION': 422, 'LEAVES': 186, 'CLEARANCE': 121, 'AOsMEMOs': 32}
Fold 4: train=2928, val=690, val_classes={'COMMUNICATION': 422, 'CLEARANCE': 122, 'LEAVES': 113, 'AOsMEMOs': 33}
Fold 5: train=2929, val=689, val_classes={'COMMUNICATION': 423, 'CLEARANCE': 121, 'LEAVES': 112, 'AOsMEMOs': 33}


In [3]:
# Compare 3 leakage-free linear pipelines on identical folds
models = {
    "tfidf_word_logreg": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "tfidf_word_char_logreg": Pipeline([
        ("features", FeatureUnion([
            ("word", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)),
            ("char", TfidfVectorizer(lowercase=True, analyzer="char_wb", ngram_range=(3, 5), min_df=2)),
        ])),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]),
    "tfidf_word_linsvc": Pipeline([
        ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=2)),
        ("clf", LinearSVC(class_weight="balanced", random_state=42)),
    ]),
}

scoring = {"accuracy": "accuracy", "f1_macro": "f1_macro"}
summary_rows = []
oof_preds = {}

for name, pipe in models.items():
    scores = cross_validate(pipe, X, y, cv=cv, groups=groups, scoring=scoring)
    preds = cross_val_predict(pipe, X, y, cv=cv, groups=groups)
    oof_preds[name] = preds

    report = classification_report(y, preds, output_dict=True, zero_division=0)
    summary_rows.append({
        "model": name,
        "mean_accuracy": round(float(scores["test_accuracy"].mean()), 4),
        "std_accuracy": round(float(scores["test_accuracy"].std()), 4),
        "mean_macro_f1": round(float(scores["test_f1_macro"].mean()), 4),
        "std_macro_f1": round(float(scores["test_f1_macro"].std()), 4),
        "AOsMEMOs_f1": round(float(report["AOsMEMOs"]["f1-score"]), 4),
        "CLEARANCE_f1": round(float(report["CLEARANCE"]["f1-score"]), 4),
        "COMMUNICATION_f1": round(float(report["COMMUNICATION"]["f1-score"]), 4),
        "LEAVES_f1": round(float(report["LEAVES"]["f1-score"]), 4),
    })

results_df = pd.DataFrame(summary_rows).sort_values("mean_macro_f1", ascending=False).reset_index(drop=True)
display(results_df)

,model,mean_accuracy,std_accuracy,mean_macro_f1,std_macro_f1,AOsMEMOs_f1,CLEARANCE_f1,COMMUNICATION_f1,LEAVES_f1
0,tfidf_word_linsvc,0.9576,0.0254,0.9291,0.0346,0.8961,0.9465,0.9756,0.9249
1,tfidf_word_char_logreg,0.9026,0.0468,0.8378,0.0879,0.9024,0.9349,0.9329,0.7371
2,tfidf_word_logreg,0.8140,0.0769,0.7015,0.1142,0.8862,0.7905,0.8808,0.4911


In [4]:
# Detailed evaluation, confusion matrix, conflict-sensitivity check, and error analysis
baseline_name = "tfidf_word_logreg"
best_name = results_df.iloc[0]["model"]

for label_tag, mname in [("Config Baseline", baseline_name), ("Best Linear Model", best_name)]:
    if label_tag == "Best Linear Model" and best_name == baseline_name:
        continue
    preds = oof_preds[mname]
    print(f"=== {label_tag}: {mname} ===")
    print(classification_report(y, preds, digits=4, zero_division=0))

y_pred = oof_preds[best_name]
labels = sorted(y.unique())
cm = pd.DataFrame(
    confusion_matrix(y, y_pred, labels=labels),
    index=[f"actual_{l}" for l in labels],
    columns=[f"pred_{l}" for l in labels],
)
print(f"Confusion Matrix ({best_name}):")
display(cm)

no_conflict = ~df["has_label_conflict"]
macro_all = f1_score(y, y_pred, average="macro")
macro_no_conf = f1_score(y[no_conflict], y_pred[no_conflict], average="macro")
acc_all = accuracy_score(y, y_pred)
acc_no_conf = accuracy_score(y[no_conflict], y_pred[no_conflict])

print(f"Accuracy (all rows):         {acc_all:.4f} | Macro-F1 (all rows):         {macro_all:.4f}")
print(f"Accuracy (no conflict rows): {acc_no_conf:.4f} | Macro-F1 (no conflict rows): {macro_no_conf:.4f}")

errors = df.assign(predicted=y_pred)
real_errors = errors[(errors["label"] != errors["predicted"]) & (~errors["has_label_conflict"])]
print(f"\nNon-conflict misclassifications ({best_name}): {len(real_errors)}")
display(real_errors[["row_id", "label", "predicted", "document_text"]].head(25))

=== Config Baseline: tfidf_word_logreg ===
               precision    recall  f1-score   support

     AOsMEMOs     0.8834    0.8889    0.8862       162
    CLEARANCE     0.8741    0.7215    0.7905       693
COMMUNICATION     0.7927    0.9910    0.8808      2111
       LEAVES     0.9016    0.3374    0.4911       652

     accuracy                         0.8170      3618
    macro avg     0.8630    0.7347    0.7621      3618
 weighted avg     0.8320    0.8170    0.7935      3618

=== Best Linear Model: tfidf_word_linsvc ===
               precision    recall  f1-score   support

     AOsMEMOs     0.9452    0.8519    0.8961       162
    CLEARANCE     0.9613    0.9322    0.9465       693
COMMUNICATION     0.9559    0.9962    0.9756      2111
       LEAVES     0.9650    0.8880    0.9249       652

     accuracy                         0.9580      3618
    macro avg     0.9569    0.9171    0.9358      3618
 weighted avg     0.9581    0.9580    0.9574      3618

Confusion Matrix (tfidf_wo

,pred_AOsMEMOs,pred_CLEARANCE,pred_COMMUNICATION,pred_LEAVES
actual_AOsMEMOs,138,0,24,0
actual_CLEARANCE,0,646,26,21
actual_COMMUNICATION,8,0,2103,0
actual_LEAVES,0,26,47,579


Accuracy (all rows):         0.9580 | Macro-F1 (all rows):         0.9358
Accuracy (no conflict rows): 0.9657 | Macro-F1 (no conflict rows): 0.9441

Non-conflict misclassifications (tfidf_word_linsvc): 112


,row_id,label,predicted,document_text
310,COMMUNICATION-312,AOsMEMOs,COMMUNICATION,MEMORANDUM TO MR. JULIUS G. CALIMARAN DIESIGNATED AS THE PHILHEALTH EPRS HANDLER AND REPRESENTATIVE
653,COMMUNICATION-657,COMMUNICATION,AOsMEMOs,"FAMILIARIZATION SESSION ON THE PROPER COMPLETION OF THE NEW STANDARDIZED PR TEMPLATE TO BE CONDUCTED ON APRIL 7-8, 2025 @ PEO CONFERENCE..."
682,COMMUNICATION-686,COMMUNICATION,AOsMEMOs,DEPARTMENT MEMORANDUM NO. 2024-0462: ADVISORY ON THE IMPLEMETATION OF THE MEDICAL ASSISTANCE TO INDIGENT AND FINANCIALLY INCAPACITATED P...
835,COMMUNICATION-840,AOsMEMOs,COMMUNICATION,"MEMORANDUM 53, 2025: WORKSHOP ON THE HARMONIZED/STREAMLINED PROCESS FLOW FROM PROCUREMENT TP PAYMENT (BATCH 1 JUNE 10, 2025 & BATCH 2 JU..."
846,COMMUNICATION-851,AOsMEMOs,COMMUNICATION,"ADMINISTRATIVE ORDER NO. 04, SERIES OF 2025: APPROVING AND ADOPTING THE JOINT RESOLUTION NO.01, SERIES OF 2025, OF THE PFC, PAO, PHO, PH..."
914,COMMUNICATION-920,AOsMEMOs,COMMUNICATION,PROVINCIAL BUDGET MEMO NO. 01-2025: BUDGET CALL ON THE PREPARATION AND SUBMISSION OF ANNUAL GENERAL FUND BUDGET PROPOSALS FOR CY 2026
1116,COMMUNICATION-1125,AOsMEMOs,COMMUNICATION,"MEMORANDUM NO. 280, S.2025 ENDORSEMENT OF REPORTS FOR THE UTILIZED LOCAL GOVERNMENT SUPPORT FUND - FA TO LGU (LGSF - FALGU) FOR REVIEW A..."
1206,COMMUNICATION-1215,AOsMEMOs,COMMUNICATION,"MEMORANDUM NO. 162 DEPARTMENT PERFORMANCE COMMITMENT REVIEW (DPCR) MEETING ON OCTOBER 21, 2025 @9:00AM AT THE COMMISSARY, CAPITOL COMPOUND"
1291,COMMUNICATION-1301,AOsMEMOs,COMMUNICATION,MEMORANDUM ORDER NO. 65 CANCELLATION OF PREVIOUS APPROVED PROPOSALS AND FOR APPROVAL OF A NEW PROPOSAL CANCELLED PROPOSALS: 1. GREEN HOU...
1554,COMMUNICATION-1568,COMMUNICATION,AOsMEMOs,"JOINT RESOLUTION NO. 1, S.2026 A JOINT RESOLUTION NO.01, S.2026 OF THE PFC, ACCOUNTING, PHO, PHRMO TITLE ""A JOINT RESOLUTION OF THE PFC,..."


In [5]:
# Fit final models on full clean dataset and save reproducible run artifacts
RUN_DIR.mkdir(parents=True, exist_ok=True)

# Save both the config baseline (tfidf_word_logreg) and best model if different
final_baseline = models["tfidf_word_logreg"].fit(X, y)
joblib.dump(final_baseline, RUN_DIR / "tfidf_word_logreg.joblib")

final_best = models[best_name].fit(X, y)
joblib.dump(final_best, RUN_DIR / "best_model.joblib")

cm_dict = {
    row: {col: int(cm.loc[row, col]) for col in cm.columns}
    for row in cm.index
}

run_meta = {
    "run_name": "tfidf_logreg_v2",
    "dataset": str(DATA_FILE),
    "n_rows": int(len(df)),
    "n_conflict_rows": int(df["has_label_conflict"].sum()),
    "class_counts": {k: int(v) for k, v in df["label"].value_counts().items()},
    "cv": {
        "method": "StratifiedGroupKFold",
        "n_splits": 5,
        "shuffle": True,
        "random_state": 42,
        "group_col": "group_id",
    },
    "results": results_df.to_dict(orient="records"),
    "best_model": best_name,
    "best_model_oof_metrics": {
        "accuracy_all": round(float(acc_all), 4),
        "macro_f1_all": round(float(macro_all), 4),
        "accuracy_no_conflict": round(float(acc_no_conf), 4),
        "macro_f1_no_conflict": round(float(macro_no_conf), 4),
        "non_conflict_error_count": int(len(real_errors)),
    },
    "confusion_matrix": cm_dict,
}

(RUN_DIR / "run.json").write_text(json.dumps(run_meta, indent=2), encoding="utf-8")
print("Saved run artifacts to:", RUN_DIR.resolve())
print(json.dumps(run_meta["best_model_oof_metrics"], indent=2))

Saved run artifacts to: C:\Users\ASUS\Documents\GitHub\intelligent-document-finetuning\models\classification\tfidf_logreg_v2
{
  "accuracy_all": 0.958,
  "macro_f1_all": 0.9358,
  "accuracy_no_conflict": 0.9657,
  "macro_f1_no_conflict": 0.9441,
  "non_conflict_error_count": 112
}
